In [1]:
code = r"""
#include <iostream>
#include <vector>
#include <climits>
#include <omp.h>
#include <chrono>
#include <cstdlib>
#include <iomanip>
#include <cmath>
#include <fstream>
#include <algorithm>
using namespace std;

using ms   = chrono::duration<double, milli>;
using Grid = vector<vector<int>>;

// ════════════════════════════════════════════════
//  UTILITY
// ════════════════════════════════════════════════

int minVertex(const vector<int>& dist, const vector<bool>& vis, int V) {
    int minVal = INT_MAX, idx = -1;
    for (int v = 0; v < V; v++)
        if (!vis[v] && dist[v] < minVal) { minVal = dist[v]; idx = v; }
    return idx;
}

Grid createGraph(int V, int density_pct, unsigned seed = 42) {
    srand(seed);
    Grid g(V, vector<int>(V, 0));
    for (int i = 0; i < V; i++)
        for (int j = 0; j < V; j++)
            if (i != j && (rand() % 100) < density_pct)
                g[i][j] = rand() % 99 + 1;
    return g;
}

bool verify(const vector<int>& a, const vector<int>& b) { return a == b; }

double amdahl(double f, int p) { return 1.0 / ((1.0 - f) + f / p); }

// ════════════════════════════════════════════════
//  1. SERIAL DIJKSTRA  O(V²)
// ════════════════════════════════════════════════

vector<int> dijkstra_serial(const Grid& g, int src) {
    int V = g.size();
    vector<int>  dist(V, INT_MAX);
    vector<bool> vis(V, false);
    dist[src] = 0;
    for (int i = 0; i < V - 1; i++) {
        int u = minVertex(dist, vis, V);
        if (u == -1) break;
        vis[u] = true;
        for (int v = 0; v < V; v++)
            if (!vis[v] && g[u][v] && dist[u] != INT_MAX
                && dist[u] + g[u][v] < dist[v])
                dist[v] = dist[u] + g[u][v];
    }
    return dist;
}

// ════════════════════════════════════════════════
//  2. PARALLEL v1 — parallel edge relaxation
//     serial minVertex (Amdahl bottleneck)
// ════════════════════════════════════════════════

vector<int> dijkstra_omp_v1(const Grid& g, int src, int nt) {
    int V = g.size();
    vector<int>  dist(V, INT_MAX);
    vector<bool> vis(V, false);
    dist[src] = 0;
    omp_set_num_threads(nt);

    for (int i = 0; i < V - 1; i++) {
        int u = minVertex(dist, vis, V);   // serial bottleneck
        if (u == -1) break;
        vis[u] = true;

        #pragma omp parallel for schedule(dynamic, 64) default(shared)
        for (int v = 0; v < V; v++) {
            if (!vis[v] && g[u][v] && dist[u] != INT_MAX) {
                int nd = dist[u] + g[u][v];
                if (nd < dist[v]) {
                    #pragma omp critical
                    { if (nd < dist[v]) dist[v] = nd; }
                }
            }
        }
    }
    return dist;
}

// ════════════════════════════════════════════════
//  3. PARALLEL v2 — parallel min-reduction
//     + parallel relaxation (reduces bottleneck)
// ════════════════════════════════════════════════

vector<int> dijkstra_omp_v2(const Grid& g, int src, int nt) {
    int V = g.size();
    vector<int>  dist(V, INT_MAX);
    vector<bool> vis(V, false);
    dist[src] = 0;
    omp_set_num_threads(nt);

    for (int i = 0; i < V - 1; i++) {

        // --- Parallel min reduction ---
        int gmin = INT_MAX, u = -1;
        #pragma omp parallel default(shared)
        {
            int lmin = INT_MAX, lidx = -1;
            #pragma omp for nowait
            for (int v = 0; v < V; v++)
                if (!vis[v] && dist[v] < lmin) { lmin = dist[v]; lidx = v; }
            #pragma omp critical
            { if (lmin < gmin) { gmin = lmin; u = lidx; } }
        }
        if (u == -1) break;
        vis[u] = true;

        // --- Parallel relaxation ---
        #pragma omp parallel for schedule(guided) default(shared)
        for (int v = 0; v < V; v++) {
            if (!vis[v] && g[u][v] && dist[u] != INT_MAX) {
                int nd = dist[u] + g[u][v];
                if (nd < dist[v]) {
                    #pragma omp critical
                    { if (nd < dist[v]) dist[v] = nd; }
                }
            }
        }
    }
    return dist;
}

// ════════════════════════════════════════════════
//  4. PARALLEL v3 — chunk partitioning
//     each thread owns a contiguous vertex range
//     for relaxation → better cache locality
// ════════════════════════════════════════════════

vector<int> dijkstra_omp_v3(const Grid& g, int src, int nt) {
    int V = g.size();
    vector<int>  dist(V, INT_MAX);
    vector<bool> vis(V, false);
    dist[src] = 0;
    omp_set_num_threads(nt);

    for (int i = 0; i < V - 1; i++) {
        // Parallel min reduction
        int gmin = INT_MAX, u = -1;
        #pragma omp parallel default(shared)
        {
            int lmin = INT_MAX, lidx = -1;
            #pragma omp for nowait schedule(static)
            for (int v = 0; v < V; v++)
                if (!vis[v] && dist[v] < lmin) { lmin = dist[v]; lidx = v; }
            #pragma omp critical
            { if (lmin < gmin) { gmin = lmin; u = lidx; } }
        }
        if (u == -1) break;
        vis[u] = true;

        // Static chunked relaxation (cache-friendly)
        #pragma omp parallel for schedule(static) default(shared)
        for (int v = 0; v < V; v++) {
            if (!vis[v] && g[u][v] && dist[u] != INT_MAX) {
                int nd = dist[u] + g[u][v];
                if (nd < dist[v]) {
                    #pragma omp critical
                    { if (nd < dist[v]) dist[v] = nd; }
                }
            }
        }
    }
    return dist;
}

// ════════════════════════════════════════════════
//  TIMING HELPER
// ════════════════════════════════════════════════

double timeIt(function<void()> fn) {
    auto t0 = chrono::high_resolution_clock::now();
    fn();
    return ms(chrono::high_resolution_clock::now() - t0).count();
}

// ════════════════════════════════════════════════
//  MAIN
// ════════════════════════════════════════════════

int main() {
    int MAX_T = omp_get_max_threads();
    vector<int> sizes        = {500, 1000, 2000, 4000};
    vector<int> thread_cnts  = {1, 2, min(4, MAX_T)};
    vector<int> densities    = {70, 30};   // dense vs sparse
    string      density_name[] = {"Dense (70%)", "Sparse (30%)"};

    cout << "\n";
    cout << "╔══════════════════════════════════════════════════════════════════╗\n";
    cout << "║   UCS645: Parallel Dijkstra — OpenMP Strengthened Analysis      ║\n";
    cout << "╚══════════════════════════════════════════════════════════════════╝\n\n";
    cout << "System: " << MAX_T << " logical thread(s) available\n\n";

    // ── Amdahl's Law table ──────────────────────────────────────────────
    cout << "── Amdahl's Law (theoretical max speedup, f=0.75 parallel) ──\n";
    cout << left << setw(10) << "Threads"
         << setw(14) << "Max Speedup" << "Efficiency\n";
    cout << string(38, '-') << "\n";
    for (int p : {1,2,4,8,16}) {
        double s = amdahl(0.75, p);
        cout << setw(10) << p
             << setw(14) << fixed << setprecision(2) << s
             << fixed << setprecision(1) << (s/p*100) << "%\n";
    }
    cout << "\n";

    // ── CSV output ──────────────────────────────────────────────────────
    ofstream csv("results.csv");
    csv << "Graph_Size,Density,Algorithm,Threads,Time_ms,Speedup,Efficiency_pct,Correct\n";

    // ── Benchmark loop ──────────────────────────────────────────────────
    for (int di = 0; di < 2; di++) {
        int density = densities[di];
        cout << "══════════════════════════════════════════\n";
        cout << "  Graph Type: " << density_name[di] << "\n";
        cout << "══════════════════════════════════════════\n\n";

        for (int V : sizes) {
            cout << "  ┌─ V = " << V << " vertices ─────────────────\n";
            auto graph = createGraph(V, density);
            vector<int> d_ref;
            double t_ser = 0;

            // Serial baseline
            {
                vector<int> d;
                t_ser = timeIt([&]{ d = dijkstra_serial(graph, 0); });
                d_ref = d;
                cout << "  │  Serial:          " << setw(9) << fixed
                     << setprecision(2) << t_ser << " ms  (baseline)\n";
                csv << V << "," << density_name[di] << ",Serial,1,"
                    << t_ser << ",1.00,100.0,YES\n";
            }

            // Parallel variants
            struct Variant { string name; function<vector<int>(int)> fn; };
            vector<Variant> variants = {
                {"OMP-v1(dynamic)", [&](int p){ return dijkstra_omp_v1(graph,0,p); }},
                {"OMP-v2(guided) ", [&](int p){ return dijkstra_omp_v2(graph,0,p); }},
                {"OMP-v3(static) ", [&](int p){ return dijkstra_omp_v3(graph,0,p); }},
            };

            for (auto& var : variants) {
                for (int p : thread_cnts) {
                    vector<int> d;
                    double t = timeIt([&]{ d = var.fn(p); });
                    double spd = t_ser / t;
                    double eff = spd / p * 100.0;
                    bool   ok  = verify(d_ref, d);
                    cout << "  │  " << var.name << "  p=" << p
                         << "  " << setw(9) << t << " ms"
                         << "  spd=" << setw(5) << spd << "x"
                         << "  eff=" << setw(6) << eff << "%"
                         << "  " << (ok ? "OK" : "WRONG!") << "\n";
                    csv << V << "," << density_name[di] << ","
                        << var.name << "," << p << ","
                        << t << "," << spd << ","
                        << eff << "," << (ok?"YES":"NO") << "\n";
                }
            }
            cout << "  └──────────────────────────────────────\n\n";
        }
    }

    csv.close();
    cout << "\n✓ Results saved to results.csv (copy to Excel for report)\n\n";

    // ── Summary insight ─────────────────────────────────────────────────
    cout << "── Key Observations ──────────────────────────────────────\n";
    cout << "  1. Speedup improves with larger V (parallelism pays off)\n";
    cout << "  2. Dense graphs benefit more than sparse (more edge work)\n";
    cout << "  3. OMP-v3 (static schedule) has better cache locality\n";
    cout << "  4. Amdahl bottleneck: serial minVertex caps speedup at ~2x\n";
    cout << "  5. Efficiency drops as p increases (synchronisation cost)\n";
    cout << "──────────────────────────────────────────────────────────\n\n";

    return 0;
}
"""

import subprocess

# Write
with open('/content/dijkstra_omp.cpp', 'w') as f:
    f.write(code)
print("File written")

# Compile
r = subprocess.run(
    ['g++', '-O2', '-fopenmp', '/content/dijkstra_omp.cpp', '-o', '/content/dijkstra_omp'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print("Compiled successfully!\n")
    out = subprocess.run(['/content/dijkstra_omp'], capture_output=True, text=True)
    print(out.stdout)
    if out.stderr: print("STDERR:", out.stderr)
else:
    print("Compile error:\n", r.stderr)

File written
Compiled successfully!


╔══════════════════════════════════════════════════════════════════╗
║   UCS645: Parallel Dijkstra — OpenMP Strengthened Analysis      ║
╚══════════════════════════════════════════════════════════════════╝

System: 2 logical thread(s) available

── Amdahl's Law (theoretical max speedup, f=0.75 parallel) ──
Threads   Max Speedup   Efficiency
--------------------------------------
1         1.00          100.0%
2         1.60          80.0%
4         2.29          57.1%
8         2.91          36.4%
16        3.37          21.1%

══════════════════════════════════════════
  Graph Type: Dense (70%)
══════════════════════════════════════════

  ┌─ V = 500 vertices ─────────────────
  │  Serial:          1.52      ms  (baseline)
  │  OMP-v1(dynamic)  p=1  2.04      ms  spd=0.74 x  eff=74.39 %  OK
  │  OMP-v1(dynamic)  p=2  1.69      ms  spd=0.90 x  eff=44.94 %  OK
  │  OMP-v1(dynamic)  p=2  1.57      ms  spd=0.97 x  eff=48.34 %  OK
  │  OMP-v2(guided)  

In [ ]:
from google.colab import files
files.download('/content/results.csv')